In [8]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm
import json
from PIL import Image
from collections import defaultdict

# ===================== 配置 =====================
IMAGE_DIR = "/root/autodl-tmp/figures"  # 混合图路径
FEATURE_SAVE_DIR = "/root/autodl-tmp/vit_new_npz_features"  # 输出：1只股票1个npz
BREAKPOINT_FILE = os.path.join(FEATURE_SAVE_DIR, "vit_breakpoint.json")
# 🔥 新增：你的原始CSV路径
CSV_PATH = "/root/autodl-tmp/日个股数据2.0.csv"

BATCH_SIZE = 32
IMAGE_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_FOLDERS = None

# 权重路径
LOCAL_CKPT = "/root/autodl-tmp/mae_model/model.safetensors"

# ===========================================================================

os.makedirs(FEATURE_SAVE_DIR, exist_ok=True)
print(f"使用设备: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU型号: NVIDIA GeForce RTX 4090 D")

# ===================== 🔥 加载真实标签：股票ID+日期 → 收益率 =====================
def load_label_mapping(csv_path):
    """从CSV构建映射：{股票ID_日期: 收益率Dretwd}"""
    df = pd.read_csv(csv_path)
    # 格式化日期为 20200901 格式
    df['Trddt'] = pd.to_datetime(df['Trddt']).dt.strftime('%Y%m%d')
    # 构建key：股票ID_日期
    df['key'] = df['Stkcd'].astype(str) + '_' + df['Trddt']
    # 返回映射字典
    return dict(zip(df['key'], df['Dretwd']))

# 全局加载标签映射（只加载一次）
LABEL_MAP = load_label_mapping(CSV_PATH)

# ===================== 标签 + 股票ID解析（修复版） =====================
def parse_stock_info(filename):
    """
    从文件名解析：股票ID + 真实收益率标签
    文件名格式：{stock_id}_{date}.png
    """
    base = os.path.splitext(filename)[0]  # 1_20200901
    stock_id = base.split("_")[0]         # 股票ID
    # 🔥 修复：从CSV取真实标签（收益率），不再用股票ID！
    label = LABEL_MAP.get(base, 0.0)      # 真实标签：Dretwd
    return stock_id, label

# ===================== 数据集（完全不变） =====================
class StockImageDataset(Dataset):
    def __init__(self, image_dir, transform=None, max_folders=None):
        self.image_dir = image_dir
        self.transform = transform
        self.valid_files = []

        all_subdirs = []
        for root, dirs, files in os.walk(image_dir):
            if root == image_dir:
                all_subdirs = [os.path.join(root, d) for d in dirs]
                break

        if max_folders is not None:
            all_subdirs = all_subdirs[:max_folders]
            print(f"试运行：仅前 {max_folders} 个文件夹")

        print("正在预检查图片有效性...")
        for d in all_subdirs:
            for root, _, files in os.walk(d):
                for f in files:
                    if f.endswith(".png"):
                        path = os.path.join(root, f)
                        try:
                            with Image.open(path) as img:
                                pass
                            self.valid_files.append(path)
                        except:
                            print(f"🗑️ 跳过坏图: {path}")

        print(f"✅ 有效图像总数: {len(self.valid_files)}")

    def __len__(self):
        return len(self.valid_files)

    def __getitem__(self, idx):
        path = self.valid_files[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

# ===================== 纯 ViT 模型（完全不变） =====================
class PureViT(torch.nn.Module):
    def __init__(self):
        super().__init__()
        import timm
        import safetensors.torch

        self.vit = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=0)
        state_dict = safetensors.torch.load_file(LOCAL_CKPT, device="cpu")
        self.vit.load_state_dict(state_dict, strict=False)

    @torch.no_grad()
    def forward(self, x):
        return self.vit(x)

# ===================== 断点续跑（完全不变） =====================
def load_breakpoint():
    if os.path.exists(BREAKPOINT_FILE):
        with open(BREAKPOINT_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_breakpoint(files):
    with open(BREAKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(list(files), f)

# ===================== 主程序：按股票聚合保存（完全不变） =====================
if __name__ == "__main__":
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.225, 0.225])
    ])

    dataset = StockImageDataset(IMAGE_DIR, transform, max_folders=MAX_FOLDERS)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model = PureViT().to(DEVICE)
    model.eval()

    processed = load_breakpoint()
    print("🚀 开始纯 ViT 特征提取（按股票聚合）...")

    # 🔥 核心：按股票ID分组存储特征+标签
    stock_data = defaultdict(lambda: {"feats": [], "labels": []})

    for imgs, names in tqdm(dataloader, desc="提取特征"):
        imgs = imgs.to(DEVICE)

        # 过滤已处理的图片
        valid_img = []
        valid_name = []
        for img, name in zip(imgs, names):
            if name not in processed:
                valid_img.append(img)
                valid_name.append(name)
        if not valid_img:
            continue

        # 批量推理
        tensor = torch.stack(valid_img)
        feats = model(tensor).cpu().numpy()

        # 按股票ID分组
        for name, feat in zip(valid_name, feats):
            stock_id, label = parse_stock_info(name)
            stock_data[stock_id]["feats"].append(feat)
            stock_data[stock_id]["labels"].append(label)
            processed.add(name)

        # 断点保存
        if len(processed) % (10 * BATCH_SIZE) == 0:
            save_breakpoint(processed)

    # ===================== 🔥 最终保存：每只股票1个NPZ（完全不变） =====================
    print("\n💾 开始按股票保存特征...")
    for stock_id, data in tqdm(stock_data.items(), desc="保存股票NPZ"):
        # 转换为numpy数组
        feat_arr = np.array(data["feats"])      # shape: [样本数, 768]
        label_arr = np.array(data["labels"])    # shape: [样本数]
        
        # 保存：1只股票 = 1个npz文件
        save_path = os.path.join(FEATURE_SAVE_DIR, f"stock_{stock_id}_vit.npz")
        np.savez(save_path, feature=feat_arr, label=label_arr)

    # 最终保存断点
    save_breakpoint(processed)

    # 输出结果
    print("\n🎉 全部完成！")
    print(f"✅ 处理图片总数：{len(processed)} 个")
    print(f"✅ 聚合股票总数：{len(stock_data)} 只")
    print(f"✅ 保存格式：1只股票 = 1个NPZ (feature+label)")
    print(f"📁 特征路径：{FEATURE_SAVE_DIR}")

使用设备: cuda
GPU型号: NVIDIA GeForce RTX 4090 D
正在预检查图片有效性...
🗑️ 跳过坏图: /root/autodl-tmp/figures/63/63_20211229.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/100/100_20211202.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/338/338_20211231.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/651/651_20211215.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/800/800_20211203.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/786/786_20211228.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/1979/1979_20211124.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/2422/2422_20220211.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/2352/2352_20211213.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/2601/2601_20211020.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300033/300033_20211216.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300059/300059_20211206.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300124/300124_20211213.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300408/300408_20211206.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300418/300418_20211222.png
🗑️ 跳过坏图: /root/autodl-tmp/figures/300433/300433_20211202.png
🗑️ 跳过坏图: /root/autod

提取特征: 100%|██████████| 9717/9717 [10:12<00:00, 15.86it/s]



💾 开始按股票保存特征...


保存股票NPZ: 100%|██████████| 270/270 [00:01<00:00, 177.57it/s]



🎉 全部完成！
✅ 处理图片总数：310929 个
✅ 聚合股票总数：270 只
✅ 保存格式：1只股票 = 1个NPZ (feature+label)
📁 特征路径：/root/autodl-tmp/vit_new_npz_features
